# GENERAR COTIZACION

# GENERAR DOCUMENTO

In [95]:
from docxtpl import DocxTemplate, RichText
from datetime import date
import re
import tempfile

_MESES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre",
]

_IF_BLOCK = re.compile(r'\{%-?\s*if\s+\w+\s*-?%\}.*\{%-?\s*endif\s*-?%\}', re.DOTALL)
_SENTINEL = "\u00abCOND\u00bb"  # «COND» — marca párrafos condicionales


def _prepare_template(template_path: str) -> str:
    """Copia el template a un temporal añadiendo el sentinel al inicio
    de cada párrafo condicional {% if %}...{% endif %}.
    Retorna la ruta del archivo temporal."""
    doc = Document(template_path)
    for para in doc.paragraphs:
        if _IF_BLOCK.search(para.text) and para.runs:
            para.runs[0].text = _SENTINEL + para.runs[0].text
    fd, tmp = tempfile.mkstemp(suffix=".docx")
    import os; os.close(fd)
    doc.save(tmp)
    return tmp


def _remove_null_paragraphs(output_path: Path):
    """Elimina párrafos donde el sentinel quedó solo (condición False)
    y limpia el sentinel de los párrafos donde la condición fue True."""
    doc = Document(output_path)
    to_remove = []
    for para in doc.paragraphs:
        if para.text.strip() == _SENTINEL:
            to_remove.append(para)
        elif _SENTINEL in para.text:
            for run in para.runs:
                if _SENTINEL in run.text:
                    run.text = run.text.replace(_SENTINEL, "")
                    break
    for para in to_remove:
        para._element.getparent().remove(para._element)
    doc.save(output_path)


def fill_template(
    data: dict,
    template_path: str = "pruebas/template_base.docx",
    output_dir: str = "pruebas",
    fotos: list[dict] | None = None,
) -> Path:
    """Rellena template_base.docx con los datos extraídos y guarda el resultado."""
    tmp_template = _prepare_template(template_path)

    try:
        tpl = DocxTemplate(tmp_template)

        hoy = date.today()
        fecha = f"{hoy.day} de {_MESES[hoy.month - 1]} de {hoy.year}"

        ciudad_dep = data.get("ciudad_departtamento") or data.get("ciudad_departamento") or "Bogotá D.C."
        ciudad = ciudad_dep.split(",")[0].strip()

        context = {
            "NOMBRE":              (data.get("nombre") or "").upper(),
            "is_male":             "Señor" if data.get("is_male", True) else "Señora",
            "direccion":           data.get("direccion", ""),
            "apartamento":         data.get("apartamento"),
            "nombre_edificio":     data.get("nombre_edificio"),
            "ciudad_departamento": ciudad_dep,
            "ciudad":              ciudad,
            "fecha":               fecha,
            "tipo_vehiculo":       data.get("tipo_vehiculo") or "Tesla",
            "fotos":               fotos or [],
            "new_page":            RichText("\f"),
        }

        tpl.render(context)

        direccion_slug = re.sub(r'[^\w\s-]', '', data.get("direccion", "documento")).strip().replace(" ", "_")
        timestamp = hoy.strftime("%Y%m%d")
        output_path = Path(output_dir) / f"{direccion_slug}_{timestamp}.docx"
        tpl.save(output_path)
    finally:
        Path(tmp_template).unlink(missing_ok=True)

    _remove_null_paragraphs(output_path)
    return output_path


In [96]:
extracted

{'nombre': 'Andrea Ladino',
 'is_male': False,
 'direccion': 'carera 26 No 50 - 47',
 'apartamento': '503',
 'nombre_edificio': None,
 'costo_total': 2100000,
 'ciudad_departamento': 'Bogotá D.C.',
 'tipo_vehiculo': 'Tesla'}

In [98]:
# Uso
output = fill_template(extracted)
print(f"Guardado en: {output}")


Guardado en: pruebas/carera_26_No_50_-_47_20260401.docx
